# Stacks and Queues

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Choose between a stack and a queue for a task, and use ArrayDeque as either
- Simulate a process that takes turns with a queue
- Find each element's next bigger value with a monotonic stack, in O(n)
- Keep a sliding window's maximum with a monotonic deque
- Evaluate an expression with operator precedence using two stacks

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check(...) prints ✓ or ✗; the exercises below use it
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": expected " + expected + ", got " + actual);
    }
}

## In short

A **stack** gives back the last value put in (last in, first out): undo history, a browser's Back button, matching brackets, the method calls of a running program. A **queue** gives back the first (first in, first out): print jobs, a CPU taking turns between programs, messages waiting to be handled.

In Java, `ArrayDeque` is both. As a stack: `push`, `pop` and `peek`. As a queue: `offer`, `poll` and `peek`. These operations at the ends take amortized O(1) time.

![A stack and a queue in an ArrayDeque](deque.svg)

#### 🎞️ A stack and a queue, step by step

![A stack: push A, push B, then pop takes B. A queue: D joins at the back, then poll takes A from the front.](stack-and-queue-steps.svg)

**One ArrayDeque, used both ways:**

In [ ]:
Deque<String> stack = new ArrayDeque<>();
stack.push("A");
stack.push("B");
stack.push("C");
IO.println("stack: " + stack + ", peek " + stack.peek() + ", pop " + stack.pop() + ", now " + stack);

Deque<String> queue = new ArrayDeque<>();
queue.offer("A");
queue.offer("B");
queue.offer("C");
IO.println("queue: " + queue + ", peek " + queue.peek() + ", poll " + queue.poll() + ", now " + queue);

### ✏️ Exercise: Reverse the words

A stack gives things back in the opposite order. Push every word of the sentence, then pop them all to build the answer.

In [ ]:
String reverseWords(String sentence) {
    Deque<String> stack = new ArrayDeque<>();
    // TODO: push every word of sentence.split(" "), then pop them into the result
    return "";
}

void main() {
    check("three words", reverseWords("one two three"), "three two one");
    check("one word", reverseWords("hello"), "hello");
    check("repeated words", reverseWords("to be or not to be"), "be to not or be to");
}

<details>
<summary>Show an answer</summary>

```java
String reverseWords(String sentence) {
    Deque<String> stack = new ArrayDeque<>();
    for (String word : sentence.split(" ")) {
        stack.push(word);
    }
    StringJoiner result = new StringJoiner(" ");
    while (!stack.isEmpty()) {
        result.add(stack.pop());
    }
    return result.toString();
}

void main() {
    check("three words", reverseWords("one two three"), "three two one");
    check("one word", reverseWords("hello"), "hello");
    check("repeated words", reverseWords("to be or not to be"), "be to not or be to");
}
```

</details>

### ✏️ Exercise: Hot potato

Children numbered 1 to n stand in a circle and pass a potato. After every k passes, the child holding it leaves. Return the last child left. A queue models the circle: passing is `offer(poll())`.

In [ ]:
int lastChild(int n, int k) {
    Deque<Integer> circle = new ArrayDeque<>();
    for (int i = 1; i <= n; i++) {
        circle.offer(i);
    }
    // TODO: while more than one child is left, pass the potato k times, then remove the child holding it
    return circle.peek();
}

void main() {
    check("one child", lastChild(1, 3), 1);
    check("5 children, 2 passes", lastChild(5, 2), 4);
    check("6 children, 1 pass", lastChild(6, 1), 5);
    check("10 children, 4 passes", lastChild(10, 4), 3);
}

<details>
<summary>Show an answer</summary>

```java
int lastChild(int n, int k) {
    Deque<Integer> circle = new ArrayDeque<>();
    for (int i = 1; i <= n; i++) {
        circle.offer(i);
    }
    while (circle.size() > 1) {
        for (int pass = 0; pass < k; pass++) {
            circle.offer(circle.poll());
        }
        circle.poll();
    }
    return circle.peek();
}

void main() {
    check("one child", lastChild(1, 3), 1);
    check("5 children, 2 passes", lastChild(5, 2), 4);
    check("6 children, 1 pass", lastChild(6, 1), 5);
    check("10 children, 4 passes", lastChild(10, 4), 3);
}
```

</details>

Two patterns turn O(n²) loops into O(n):

- A **monotonic stack** keeps its values in decreasing order. When a bigger value arrives, it pops every smaller one, and each popped value has just found its next bigger value. Every value is pushed once and popped at most once, so the loop inside the loop costs O(n) in total.
- A **monotonic deque** does the same for a sliding window: its front is the window's maximum, values that can never be the maximum again leave from the back, and values that leave the window leave from the front.

**A monotonic stack, step by step:** for each value, the nearest smaller value to its left. Watch the stack stay increasing.

In [ ]:
int[] values = {4, 7, 3, 5, 8, 2, 6};
Deque<Integer> stack = new ArrayDeque<>();
for (int v : values) {
    while (!stack.isEmpty() && stack.peek() >= v) {
        stack.pop(); // can never be the answer for anything to the right of v
    }
    IO.println(v + ": nearest smaller on the left = " + (stack.isEmpty() ? "none" : stack.peek()) + "   stack (top first) " + stack);
    stack.push(v);
}

Two stacks, one for numbers and one for operators, also evaluate expressions like `2 + 3 * 4` with the right precedence.

### ✏️ Exercise: Simplify a path

In a folder path, `..` goes up one folder and `.` stays put. Split the path at `/`, push each folder name on a stack, pop on `..`, and skip `.` and empty parts. Then join what is left.

In [ ]:
String simplify(String path) {
    Deque<String> folders = new ArrayDeque<>();
    // TODO: go through path.split("/"), using folders as a stack
    return "/";
}

void main() {
    check("up one folder", simplify("/a/b/../c"), "/a/c");
    check("dots and slashes", simplify("/a/./b//c/"), "/a/b/c");
    check("above the root", simplify("/../x"), "/x");
    check("back to the root", simplify("/a/.."), "/");
}

<details>
<summary>Show an answer</summary>

```java
String simplify(String path) {
    Deque<String> folders = new ArrayDeque<>();
    for (String part : path.split("/")) {
        if (part.equals("..")) {
            folders.pollFirst(); // nothing happens above the root
        } else if (!part.isEmpty() && !part.equals(".")) {
            folders.push(part);
        }
    }
    StringBuilder result = new StringBuilder();
    while (!folders.isEmpty()) {
        result.append('/').append(folders.pollLast()); // the bottom of the stack is the first folder
    }
    return result.length() == 0 ? "/" : result.toString();
}

void main() {
    check("up one folder", simplify("/a/b/../c"), "/a/c");
    check("dots and slashes", simplify("/a/./b//c/"), "/a/b/c");
    check("above the root", simplify("/../x"), "/x");
    check("back to the root", simplify("/a/.."), "/");
}
```

</details>

Watch out: `pop()` on an empty `ArrayDeque` throws an exception, while `poll()` returns `null`.

**pop() and poll() on an empty deque:**

In [ ]:
Deque<Integer> empty = new ArrayDeque<>();
IO.println("poll(): " + empty.poll());
try {
    empty.pop();
} catch (NoSuchElementException e) {
    IO.println("pop(): " + e.getClass().getSimpleName());
}

## Readings

- [Stacks and Queues](https://dev.java/learn/api/collections-and-streams/collections-framework/stacks-queues/) (dev.java (Oracle), 6 min): The method tables: which methods throw an exception on an empty (or full) deque and which return null or false, and why ArrayDeque replaces the old Stack class.
- [Stacks](https://usaco.guide/gold/stacks) (USACO Guide, 10 min): "Nearest Smaller Element": the stack keeps only values that can still be an answer, which makes it monotonic. Warmer Days is the same idea with "bigger" and to the right. Pick Java in the site's settings to see Java code.
- [Sliding Window](https://usaco.guide/gold/sliding-window) (USACO Guide, 15 min): "Sliding Window Maximum in O(N)", Method 1 (Deque): values leave the back when a bigger one arrives and the front when they leave the window. Skip "With Two Pointers" (unit 1 covered it) and the problem lists.

## Check yourself

1. Which structure does a Back button need, and which a print queue?
2. Why is a monotonic stack O(n), although it has a loop inside a loop?
3. In `8 - 3 - 2`, which subtraction must be done first, and why?

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Round-Robin Scheduler** · Easy
- **Warmer Days** · Medium
- **Peak Load** · Medium
- **Calculator** · Hard